In [1]:
from utils import *
from max_cover import *
from max_cut import *
from max_cut_weighted import *
from imm import *
from knapsack_imm import knapsack_greedy
from heuristic_description import heuristic_description





problem = "Maximum Coverage"
budget = 100
dataset = "HK"
iterations = 2

if problem == "Maximum Coverage":
    heuristic = greedy_max_cover
elif problem == "Maximum Coverage Weighted":
    heuristic = knapsack_greedy_max_cover


elif problem == "Influence Maximization":
    heuristic = imm
elif problem == "Influence Maximization Weighted":
    heuristic = knapsack_greedy  
elif problem == "Maximum Cut":
    heuristic = maxcut_greedy
elif problem == "Maximum Cut Weighted":
    heuristic = DLA
else:
    raise ValueError(f"Unknown problem: {problem}")


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


results = []


train_graph = load_from_pickle(f'../snap_dataset/train/{dataset}')
val_graph = load_from_pickle(f'../snap_dataset/val/{dataset}')
test_graph = load_from_pickle(f'../snap_dataset/test/{dataset}')

print(f'number of nodes in train graph: {train_graph.number_of_nodes()}')
print(f'number of nodes in val graph: {val_graph.number_of_nodes()}')
print(f'number of nodes in test graph: {test_graph.number_of_nodes()}')

if problem in ["Maximum Coverage Weighted", "Influence Maximization Weighted", "Maximum Cut Weighted"]:
    train_graph = assign_normalized_degree_weights(train_graph)
    val_graph = assign_normalized_degree_weights(val_graph)
    test_graph = assign_normalized_degree_weights(test_graph)


explainer_feedback_list = []
history = []
best_score = float('-inf')

save_folder = f"{problem}/{dataset}"
os.makedirs(save_folder, exist_ok=True)

model_save_path = os.path.join(save_folder, "best_model.pth")
best_model_data_path = os.path.join(save_folder, "best_model_data.pkl")
history_path = os.path.join(save_folder, "history.pkl")



for iter in tqdm(range(iterations)):
    print(f"Feature Space Search Iteration {iter+1} for problem: {problem}")

    cumulative_feedback = "\n".join(explainer_feedback_list) if explainer_feedback_list else None

    if cumulative_feedback:

        summary_prompt = generate_summary_prompt(cumulative_feedback= cumulative_feedback)
        summary = get_response(client, summary_prompt)
    else:
        summary = None


    node_feature_prompt = generate_llm_prompt(
        problem = problem,
        heuristic_description=heuristic_description[problem],
        problem_definition = problem_definitions[problem],
        explainer_feedback = summary
    )
    
    proposed_features = get_response(client,node_feature_prompt)

    try:
        features, definitions, reasons = parse_llm_features(proposed_features)
        print(f"Proposed features: {features}")
    except Exception as e:
        print(f"Error parsing LLM features: {e}")
        continue

    try:
        train_features, codes = generate_train_features(problem = problem,
                                                        features= features, 
                                                        definitions= definitions, 
                                                        train_graph= train_graph, 
                                                        test_graph= test_graph,
                                                        budget=budget, 
                                                        timeout= 5
                                                        )
    except Exception as e:
        print(f"Error generating train features: {e}")
        continue

    if len(codes) == 0:
        print("No valid features generated. Skipping iteration.")
        continue
    
    model, ratio, size_reduction, explainer_feedback_new = train_test_evaluate_gnn(
        train_features = train_features, 
        train_graph = train_graph,
        test_graph = val_graph,
        codes = codes,
        heuristic = heuristic,
        budget = budget
    )

    multiplication = ratio * size_reduction



    if multiplication > best_score:
        best_score = multiplication
        best_model_data = {
            "iteration": iter + 1,
            "codes": codes,
            "ratio": ratio,
            "size_reduction": size_reduction,
            "multiplication": multiplication,
        }
        save_as_pickle(best_model_data, best_model_data_path)
        torch.save(model.state_dict(), model_save_path)

    history.append({
        "iteration": iter + 1,
        "features": list(codes.keys()),
        "ratio": ratio,
        "size_reduction": size_reduction,
        "explainer_feedback": explainer_feedback_new
    })
    
    explainer_feedback_list.append(
        f"Iteration {iter+1} | Ratio: {ratio}, Size reduction: {size_reduction}, "
        f"Feature importance: {explainer_feedback_new}"
    )



save_as_pickle(history, history_path)

print(f"Best model, data, and history saved in '{save_folder}'.")

    

    












/home/grads/a/anath/LLM_Pruning/utils.py:107: SyntaxWarning: invalid escape sequence '\('
  "The problem is defined over a graph \( G = (V, E) \).\n"
/home/grads/a/anath/LLM_Pruning/utils.py:108: SyntaxWarning: invalid escape sequence '\('
  "Given a budget \( k \), the objective is to select a subset of nodes \( S \subseteq V \) such that:\n"
/home/grads/a/anath/LLM_Pruning/utils.py:123: SyntaxWarning: invalid escape sequence '\('
  "The problem is defined over an undirected graph \( G = (V, E) \).\n"
/home/grads/a/anath/LLM_Pruning/utils.py:124: SyntaxWarning: invalid escape sequence '\('
  "The goal is to partition the vertex set \( V \) into two disjoint subsets \( S \) and \( V \\setminus S \)\n"
/home/grads/a/anath/LLM_Pruning/utils.py:125: SyntaxWarning: invalid escape sequence '\('
  "such that the number of edges crossing the cut, i.e., with one endpoint in \( S \) and the other in \( V \\setminus S \),\n"
/home/grads/a/anath/LLM_Pruning/utils.py:144: SyntaxWarning: invalid es

number of nodes in train graph: 10000
number of nodes in val graph: 10000
number of nodes in test graph: 2000000


  0%|          | 0/2 [00:00<?, ?it/s]

Feature Space Search Iteration 1 for problem: Maximum Coverage
Proposed features: ['closed_neighborhood_size', 'degree', 'eigenvector_centrality', 'candidate_overlap_count', 'local_clustering_coefficient']


Code for feature 'closed_neighborhood_size':
def extract_feature(G):
    import numpy as np
    nodes = list(G.nodes())
    has_loop = False
    for n in nodes:
        if G.has_edge(n, n):
            has_loop = True
            break
    if not has_loop:
        deg = dict(G.degree())
        return np.array([deg[v] + 1 for v in nodes], dtype=int)
    else:
        res = []
        neigh = G.neighbors
        for v in nodes:
            s = set(neigh(v))
            s.add(v)
            res.append(len(s))
        return np.array(res, dtype=int)



...

Training GNN


100%|██████████| 999/999 [00:01<00:00, 770.64it/s]


Objective value: 5388
A candidate node set has been provided.
Size of the candidate set = 6917
Objective value pruned: 5388
Ratio 1.0
queries ratio 0.6901663233003367
Explaining GNN predictions


100%|██████████| 2/2 [04:34<00:00, 137.33s/it]

Data has been saved to Maximum Coverage/HK/best_model_data.pkl
Data has been saved to Maximum Coverage/HK/history.pkl
Best model, data, and history saved in 'Maximum Coverage/HK'.


In [2]:
load_from_pickle('Maximum Coverage/HK/best_model_data.pkl')

{'iteration': 2,
 'codes': {'closed_neighborhood_size': 'import numpy as np\n\ndef extract_feature(G):\n    nodes = list(G.nodes())\n    out = np.empty(len(nodes), dtype=int)\n    for i, v in enumerate(nodes):\n        nbrcount = sum(1 for _ in G.neighbors(v))\n        out[i] = nbrcount if G.has_edge(v, v) else nbrcount + 1\n    return out',
  'degree': 'def extract_feature(G):\n    import numpy as np\n    deg = dict(G.degree())\n    return np.array([deg[n] for n in G], dtype=int)',
  'neighbor_degree_mean': 'def extract_feature(G):\n    import numpy as np\n    deg = dict(G.degree())\n    nodes = list(G.nodes())\n    feature = np.zeros(len(nodes), dtype=float)\n    for i, v in enumerate(nodes):\n        dv = deg[v]\n        if dv == 0:\n            feature[i] = 0.0\n        else:\n            s = 0.0\n            for u in G.neighbors(v):\n                s += deg[u]\n            feature[i] = s / dv\n    return feature'},
 'ratio': 1.0,
 'size_reduction': 0.3083,
 'multiplication': 0.30

In [4]:


problem = "Maximum Coverage"
budget = 100
dataset = "HK"
iterations = 1

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
train_graph = load_from_pickle(f'../snap_dataset/train/{dataset}')
val_graph = load_from_pickle(f'../snap_dataset/val/{dataset}')

In [7]:
explainer_feedback_list = []
history = []
best_score = float('-inf')

save_folder = f"{problem}/{dataset}"
os.makedirs(save_folder, exist_ok=True)

model_save_path = os.path.join(save_folder, "best_model.pth")
best_model_data_path = os.path.join(save_folder, "best_model_data.pkl")
history_path = os.path.join(save_folder, "history.pkl")

In [8]:
for iter in tqdm(range(iterations)):
    print(f"Feature Space Search Iteration {iter+1} for problem: {problem}")

    cumulative_feedback = "\n".join(explainer_feedback_list) if explainer_feedback_list else None

    if cumulative_feedback:

        summary_prompt = generate_summary_prompt(cumulative_feedback= cumulative_feedback)
        summary = get_response(client, summary_prompt)
    else:
        summary = None


    node_feature_prompt = generate_llm_prompt(
        problem = problem,
        heuristic_description=heuristic_description[problem],
        problem_definition = problem_definitions[problem],
        explainer_feedback = summary
    )
    
    proposed_features = get_response(client,node_feature_prompt)

    try:
        features, definitions, reasons = parse_llm_features(proposed_features)
        print(f"Proposed features: {features}")
    except Exception as e:
        print(f"Error parsing LLM features: {e}")
        continue



  0%|          | 0/1 [00:00<?, ?it/s]

Feature Space Search Iteration 1 for problem: Maximum Coverage


100%|██████████| 1/1 [00:46<00:00, 46.57s/it]

Proposed features: ['degree', 'closed_neighborhood_size', 'degree_percentile_global', 'degree_percentile_component', 'component_size', 'is_isolated', 'leaf_neighbor_count', 'hub_neighbor_ratio', 'neighbor_degree_mean', 'neighbor_degree_std', 'neighbor_degree_max', 'clustering_coefficient', 'core_number', 'page_rank', 'eigenvector_centrality', 'betweenness_centrality', 'neighborhood_edge_density', 'average_jaccard_with_neighbors', 'overlap_sum_with_neighbors', 'redundancy_index', 'min_jaccard_with_top_k_by_degree', 'mean_jaccard_with_top_k_by_degree', 'unique_coverage_vs_top_k_by_degree', 'distance_to_nearest_hub', 'two_hop_reach', 'neighbor_neighborhood_union_size', 'k_scaled_closed_neighborhood', 'within_component_rank', 'bridge_endpoint_flag', 'local_set_cover_proxy']


In [9]:
definitions

{'degree': 'Number of neighbors of v (|N(v)|).',
 'closed_neighborhood_size': 'Size of v’s closed neighborhood (|N[v]| = 1 + |N(v)|).',
 'degree_percentile_global': 'Percentile rank of v’s degree among all nodes in V.',
 'degree_percentile_component': 'Percentile rank of v’s degree within its connected component.',
 'component_size': 'Number of nodes in v’s connected component.',
 'is_isolated': 'Indicator 1 if degree(v) = 0, else 0.',
 'leaf_neighbor_count': 'Number of neighbors u of v with degree(u) = 1.',
 'hub_neighbor_ratio': 'Fraction of neighbors u with degree(u) ≥ 95th-percentile degree in V.',
 'neighbor_degree_mean': 'Average degree of v’s neighbors.',
 'neighbor_degree_std': 'Standard deviation of degrees of v’s neighbors.',
 'neighbor_degree_max': 'Maximum degree among v’s neighbors.',
 'clustering_coefficient': 'Fraction of possible edges among neighbors of v that actually exist.',
 'core_number': 'k-core index of v (maximum k such that v is in the k-core).',
 'page_rank':

In [19]:

timeout = 5 # seconds
class TimeoutException(Exception):
        pass

def handler(signum, frame):
    raise TimeoutException

signal.signal(signal.SIGALRM, handler)

train_X = []
codes = {}


if problem.endswith('Weighted'):
    graph_description = (
        f"The input is a weighted NetworkX graph `G` where each node has an attribute `'weight'`, "
        f"and an integer variable `budget` is provided.\n"
    )
    additional_description = "If the feature involves weight, use the existing `'weight'` attribute directly without recomputing it from other functions"
else:
    graph_description = (
        f"The input is a NetworkX graph `G` and the graph is undirected."
       
    )
    additional_description = ''

# Add tqdm to loop
for idx, feature in enumerate(tqdm(features, desc="Extracting features", unit="feature")):
    prompt_code = (
        f"{graph_description}"
        f"Feature name: '{feature}'\n"
        # f"Feature definition: '{definitions[feature]}'\n"
        # f"Write Python code for a function `extract_feature(G, budget)` that computes this feature for all nodes in `G`. "
        f"Write Python code for a function `extract_feature(G)` that computes this feature for all nodes in `G`. "
        f"{additional_description}"
        f"If the budget is relevant to the computation, incorporate it. "
        f"The function should return a NumPy array with the computed feature values.\n"
        f"Ensure the code is efficient and avoids expensive computations.\n"
        f"DO NOT INCLUDE ANY EXPLANATIONS OR COMMENTS.\n"
    )



    start = time.time()
    code_response = get_response(client,prompt_code)
    code = clean_code_block(code_response)

    print(f"Code for feature '{feature}':\n{code}\n")

    
    end = time.time()
    # print(f"Code for feature '{feature}' generated in {end - start:.2f} seconds")

    try:
        # print(f"Extracting feature '{feature}'")
        signal.alarm(timeout)  # Set timeout

        namespace = {}
        exec(code, namespace)  # Execute code in namespace

        # namespace["extract_feature"](G=test_graph, budget=budget)
        feature_values = namespace["extract_feature"](G=train_graph)



        if  isinstance(feature_values, np.ndarray) and feature_values.shape[0] == train_graph.number_of_nodes():
            # train_X[feature] = feature_values
            train_X.append(feature_values)

            codes[feature] = code
            # codes.append(code)
    except (TimeoutException, Exception) as e:

        
        print(f"⚠️ Skipping feature '{feature}' due to error: {e}")

        print('*'*30)
        print(code)
        print('*'*30)
    finally:
        signal.alarm(0)  # Reset alarm


    break



Extracting features:   0%|          | 0/30 [00:00<?, ?feature/s]

Code for feature 'degree':
import numpy as np

def extract_feature(G):
    return np.fromiter((d for _, d in G.degree()), dtype=np.int64, count=G.number_of_nodes())



Extracting features:   0%|          | 0/30 [00:09<?, ?feature/s]


In [16]:
namespace = {}
exec(code, namespace)  # Execute code in namespace

# namespace["extract_feature"](G=test_graph, budget=budget)
feature_values = namespace["extract_feature"](G=train_graph)

In [18]:
if  isinstance(feature_values, np.ndarray) and feature_values.shape[0] == train_graph.number_of_nodes():
    # train_X[feature] = feature_values
    train_X.append(feature_values)

    codes[feature] = code

TypeError: list indices must be integers or slices, not str